In [1]:
import pandas as pd
import numpy as np
import os
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image
import torch.nn as nn
import torch.optim as optim
import time
from sklearn.metrics import roc_auc_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import random
# set seeds for reproducibility
random.seed(426)
np.random.seed(426)
torch.manual_seed(426)
torch.cuda.manual_seed(426)
torch.cuda.manual_seed_all(426)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
data_entry = pd.read_csv("../Datasets/archive/Data_Entry_2017.csv")

In [4]:
data_entry.shape

(112120, 12)

In [5]:
data_entry.columns

Index(['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11'],
      dtype='object')

In [6]:
data_entry.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [7]:
# locate where the pictures are in so we can load it into the environment for filtering
all_folders = [
    "../Datasets/archive/images_001/images",
    "../Datasets/archive/images_002/images",
    "../Datasets/archive/images_003/images",
    "../Datasets/archive/images_004/images",
    "../Datasets/archive/images_005/images",
    "../Datasets/archive/images_006/images",
    "../Datasets/archive/images_007/images",
    "../Datasets/archive/images_008/images",
    "../Datasets/archive/images_009/images",
]

In [8]:
# get all filenames that end with png
available_files = set()
for folder in all_folders:
    for f in os.listdir(folder):
        if f.lower().endswith(".png"):
            available_files.add(f)

In [9]:
len(available_files)

84999

In [10]:
# ensure we just take patients who are in the folders we want, since we are not using all 12 folders
df = data_entry[data_entry["Image Index"].isin(available_files)]

In [11]:
# ensure all view positions are PA or AP since those are what DenseNet was trained on
df["View Position"].unique()

array(['PA', 'AP'], dtype=object)

In [12]:
# get unique patients so we can shuffle data
all_patients = df["Patient ID"].unique()

In [13]:
# set seed for data shuffle and shuffle dataset
rng = np.random.default_rng(426)
rng.shuffle(all_patients)

In [14]:
# init split amts
n = len(all_patients)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

In [15]:
# split on shuffled dataset of unique students
train_patients = set(all_patients[:n_train])
val_patients = set(all_patients[n_train:n_train + n_val])
test_patients = set(all_patients[n_train + n_val:])

In [16]:
# get data on patients who were just split
df_train = df[df["Patient ID"].isin(train_patients)].copy()
df_val = df[df["Patient ID"].isin(val_patients)].copy()
df_test = df[df["Patient ID"].isin(test_patients)].copy()

In [17]:
# verify split worked
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    healthy  = split["Finding Labels"].eq("No Finding").sum()
    diseased = len(split) - healthy
    print(f"{name}: {len(split)} total | Healthy: {healthy} | Diseased: {diseased}")

Train: 60032 total | Healthy: 32597 | Diseased: 27435
Val: 13122 total | Healthy: 7123 | Diseased: 5999
Test: 11845 total | Healthy: 6469 | Diseased: 5376


In [18]:
# create binary labels to data since we aren't predicting all classifiers
def add_binary_label(df):
    df = df.copy()
    df["binary_label"] = np.where(
        df["Finding Labels"].str.strip() == "No Finding", 0, 1
    )
    return df

In [19]:
# create binary labels
df_train = add_binary_label(df_train)
df_val = add_binary_label(df_val)
df_test = add_binary_label(df_test)

In [20]:
# create class to read in data for dataloaders
class CSVDataset(Dataset):
    def __init__(self, df, image_folders, transform = None):
        self.df = df.reset_index(drop = True) # ensure indices are reset for proper indexing later
        self.transform = transform # store transformers to apply to each image
        
        # build dictionary mapping the filename to the full path
        self.path_map = {}
        for folder in image_folders:
            for f in os.listdir(folder):
                if f.lower().endswith(".png"):
                    self.path_map[f] = os.path.join(folder, f)
    
    def __len__(self): # dataloader requirement for length of dataset
        return len(self.df)
    
    def __getitem__(self, idx): # dataloader requirement for getting data
        
        row = self.df.iloc[idx]
        
        # looks up the file name, opens the image, and converts to RGB
        # note this should be grayscale BUT the DenseNet model requires 3 channels
        img = Image.open(self.path_map[row["Image Index"]]).convert("RGB") 
        label = torch.tensor(row["binary_label"], dtype=torch.long) # fetches binary label
        
        if self.transform:
            img = self.transform(img) # applies transforms to image
        
        return img, label

In [21]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_ds = CSVDataset(df_train, all_folders, transform=train_tfms)
val_ds = CSVDataset(df_val, all_folders, transform=eval_tfms)
test_ds = CSVDataset(df_test, all_folders, transform=eval_tfms)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 60032 | Val: 13122 | Test: 11845


In [22]:
pin = torch.cuda.is_available()

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=0, pin_memory=pin)

In [23]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt      = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

In [24]:
from torchvision.models import densenet121


model = densenet121(weights="IMAGENET1K_V1")
model.classifier = nn.Linear(1024, 2)
model = model.to(device)

criterion = FocalLoss(gamma=2)
optimizer = optim.AdamW(model.parameters(), lr=0.00005, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [25]:
from sklearn.metrics import roc_auc_score

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct      += outputs.argmax(1).eq(labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    all_probs    = []
    all_labels   = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels)
            probs   = torch.softmax(outputs, dim=1)[:, 1]  # prob of diseased class

            running_loss += loss.item() * images.size(0)
            correct      += outputs.argmax(1).eq(labels).sum().item()
            total        += labels.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc  = 100.0 * correct / total
    epoch_auc  = roc_auc_score(all_labels, all_probs)

    return epoch_loss, epoch_acc, epoch_auc

In [26]:
import copy

NUM_EPOCHS          = 15
EARLY_STOP_PATIENCE = 3

train_losses, train_accs         = [], []
val_losses,   val_accs, val_aucs = [], [], []

best_val_auc          = -float("inf")
best_epoch            = -1
best_state            = None
epochs_without_improve = 0

print("Starting training...")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12} {'Val AUC':<12} {'Time':<10}")
print("-" * 80)

total_train_start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    train_loss, train_acc          = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc,   val_auc = evaluate(model, val_loader, criterion, device)
    
    epoch_time = time.time() - epoch_start
    
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc           = val_auc
        best_epoch             = epoch + 1
        best_state             = copy.deepcopy(model.state_dict())
        epochs_without_improve = 0
    else:
        epochs_without_improve += 1

    print(f"{epoch+1:<8} {train_loss:<12.4f} {train_acc:<12.2f} {val_loss:<12.4f} {val_acc:<12.2f} {val_auc:<12.4f} {epoch_time:<10.1f}s")

    if epochs_without_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}.")
        break

total_train_time = time.time() - total_train_start
print(f"\nTotal training time: {total_train_time/60:.2f} minutes")

Starting training...
Epoch    Train Loss   Train Acc    Val Loss     Val Acc      Val AUC      Time      
--------------------------------------------------------------------------------
1        0.1590       66.56        0.1530       68.61        0.7436       1553.4    s
2        0.1485       69.98        0.1508       69.83        0.7488       1420.9    s
3        0.1429       71.85        0.1505       69.75        0.7539       1423.2    s
4        0.1361       73.65        0.1513       70.17        0.7580       1426.6    s
5        0.1292       75.33        0.1553       69.47        0.7461       1422.8    s
6        0.1187       78.30        0.1660       69.39        0.7486       1373.8    s
7        0.0988       83.36        0.1750       68.73        0.7407       1334.4    s

Early stopping triggered at epoch 7.

Total training time: 165.92 minutes


In [58]:
import json

# load best checkpoint
model.load_state_dict(best_state)

# evaluate on test set
test_start = time.time()
test_loss, test_acc, test_auc = evaluate(model, test_loader, criterion, device)
test_time = time.time() - test_start

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f} | Test AUC: {test_auc:.4f}")
print(f"Test evaluation time: {test_time:.2f} seconds")

# save model weights
torch.save(model.state_dict(), "densenet_best.pth")

# save history as JSON (avoids the torch/matplotlib conflict we ran into earlier)
history = {
    "train_losses":      train_losses,
    "train_accs":        train_accs,
    "val_losses":        val_losses,
    "val_accs":          val_accs,
    "val_aucs":          val_aucs,
    "best_val_auc":      best_val_auc,
    "best_epoch":        best_epoch,
    "test_loss":         test_loss,
    "test_acc":          test_acc,
    "test_auc":          test_auc,
    "total_train_time":  total_train_time,
    "test_time":         test_time,
}
with open("densenet_history.json", "w") as f:
    json.dump(history, f)

# save predictions
def get_predictions(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs  = torch.softmax(logits, dim=1)
            preds  = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

train_preds, train_labels, train_probs = get_predictions(model, train_loader, device)
val_preds,   val_labels,   val_probs   = get_predictions(model, val_loader,   device)
test_preds,  test_labels,  test_probs  = get_predictions(model, test_loader,  device)

np.save("densenet_predictions.npy", {
    "train": {"preds": train_preds, "labels": train_labels, "probs": train_probs},
    "val":   {"preds": val_preds,   "labels": val_labels,   "probs": val_probs},
    "test":  {"preds": test_preds,  "labels": test_labels,  "probs": test_probs},
})

Test Loss: 0.1495 | Test Acc: 70.94 | Test AUC: 0.7627
Test evaluation time: 199.00 seconds
